# Sprint 6 - Oversampling PlantDoc in Mixed Training (v7 `mixed_upsampled`)

**Goal (see AGENT.md / final_brief_and_plan.md):** Sprint 5's `mixed` kept both domains (PlantVillage
F1 0.9592, PlantDoc F1 0.4107) but PlantDoc lagged the pure fine-tune (`both` 0.5578), because
PlantDoc is only ~5% of each mixed epoch. This sprint gives field photos a bigger voice:

1. **Variant 7 `mixed_upsampled`** - same mixed training as Sprint 5, but PlantDoc's train set is
   repeated **8x** inside the loader (`--plantdoc-repeat 8`), so ~28% of every epoch is field photos.
2. Checks the **two gates** again (PlantDoc F1 > 0.1116 baseline, PlantVillage F1 >= 0.855) **plus**
   the real target: does PlantDoc climb toward `both`'s 0.5578 while PlantVillage stays ~0.96?

No augmentation variant this sprint - augmentation has now hurt field scores twice (`mixed_aug` <
`mixed`, and Sprint 3's aug < baseline), so we skip it.


### Where we are (real runs, in the CSV)
- baseline PlantVillage 0.9501 F1 | baseline PlantDoc 0.1116 F1 (the gap)
- `both` (fine-tune only) PlantDoc **0.5578** F1 but forgot lab (0.3168)
- `mixed` (both together) PlantDoc 0.4107 F1 | PlantVillage **0.9592** F1 - no forgetting


In [ ]:
import platform
import subprocess
import sys

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
try:
    gpu = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=30)
    print(gpu.stdout.strip().splitlines()[0] if gpu.stdout.strip() else gpu.stderr.strip() or "No GPU detected (CPU only)")
except Exception as exc:
    print("GPU check skipped:", exc)

## Step 1 - Mount Drive + clone repo

Requires the Sprint 0 archives on Drive and the Sprint 1 checkpoint `best_plantvillage_stage1.pt`
(warm start). The CSV from Sprints 4-5 must already be in `folium/results/` (it is, if you ran them).


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

REPO_URL = "https://github.com/io-PEAK/folium.git"   # change if you forked
REPO_DIR = Path("/content/folium")
DATA_DIR = Path("/content/drive/MyDrive/folium/data")    # durable archives (from Sprint 0)
LOCAL_RAW_DIR = Path("/content/folium_raw")             # per-session raw
LOCAL_DATA_DIR = Path("/content/folium_data")            # per-session organized splits
CHECKPOINT_DIR = Path("/content/drive/MyDrive/folium/checkpoints")
RESULTS_DIR = Path("/content/drive/MyDrive/folium/results")

if not (REPO_DIR / "ml").exists():
    %cd /content
    !git clone --depth 1 {REPO_URL}
else:
    !git -C {REPO_DIR} pull --ff-only -q

for d in (DATA_DIR, LOCAL_RAW_DIR, LOCAL_DATA_DIR, CHECKPOINT_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("DATA_DIR (archives):", DATA_DIR)
print("LOCAL_DATA_DIR:", LOCAL_DATA_DIR)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)
print("RESULTS_DIR:", RESULTS_DIR)

## Step 2 - Install dependencies

In [ ]:
%pip install -q --upgrade pip
%pip install -q torch torchvision albumentations matplotlib pandas tqdm opencv-python-headless scikit-learn

## Step 3 - Hydrate raw from Drive, then organize splits locally

Same as every sprint: unzip both archives, build `train/val/test` folders (seed 42, deterministic)
plus `class_map.json`. Idempotent.


In [ ]:
import sys

sys.path.insert(0, str(REPO_DIR))
from scripts.download_datasets import PLANTVILLAGE_EXPECTED, PLANTDOC_EXPECTED, hydrate_dataset

for name, expected in (("plantvillage", PLANTVILLAGE_EXPECTED), ("plantdoc", PLANTDOC_EXPECTED)):
    try:
        hydrate_dataset(LOCAL_RAW_DIR, DATA_DIR, name, expected)
    except RuntimeError as exc:
        print("HYDRATE FAILED:", exc)
        raise

result = subprocess.run([
    sys.executable,
    str(REPO_DIR / "scripts" / "organize_datasets.py"),
    "--raw-dir", str(LOCAL_RAW_DIR),
    "--data-dir", str(LOCAL_DATA_DIR),
], cwd=str(REPO_DIR))
assert result.returncode == 0, "organize_datasets.py failed"
print("splits ready at", LOCAL_DATA_DIR)

## Step 4 - Train variant 7 (`mixed_upsampled`)

Identical to Sprint 5's `mixed` **except** `--plantdoc-repeat 8`: the PlantDoc train set (2,107
mapped images) is repeated 8x inside the mixed loader, so each epoch is
43,429 PV + 16,856 PD = 60,285 images (~28% field photos instead of ~5%). Warm-started from the
Sprint 1 baseline, head-only, no augmentation. Artifacts:
`plantvillage_mixed_upsampled_epochNN.pt` + `best_plantvillage_mixed_upsampled.pt`.

The epoch print header should show `Training head-only on plantvillage+plantdoc x8 (60285 train
images, 38 classes)`.


In [ ]:
cmd = [
    sys.executable, "-m", "ml.train",
    "--data-dir", str(LOCAL_DATA_DIR),
    "--dataset", "plantvillage",
    "--mix-with", "plantdoc",
    "--plantdoc-repeat", "8",
    "--init-from", str(CHECKPOINT_DIR / "best_plantvillage_stage1.pt"),
    "--lr", "1e-3",
    "--head-lr", "1e-3",
    "--epochs", "10",
    "--batch-size", "32",
    "--checkpoint-dir", str(CHECKPOINT_DIR),
    "--num-workers", "2",
    "--tag", "mixed_upsampled",
]
result = subprocess.run(cmd, cwd=str(REPO_DIR))
assert result.returncode == 0, "ml.train failed"
print("best checkpoint:", CHECKPOINT_DIR / "best_plantvillage_mixed_upsampled.pt")

## Step 5 - Evaluate variant 7 on BOTH test sets

Two rows: `mixed_upsampled` on `plantdoc_test` (did PlantDoc climb toward `both`'s 0.5578?) and on
`plantvillage_test` (did PlantVillage stay ~0.96?).


In [ ]:
for dataset, extra in [("plantdoc", ["--map-to-pv"]), ("plantvillage", [])]:
    cmd = [
        sys.executable, "-m", "ml.evaluate",
        "--checkpoint", str(CHECKPOINT_DIR / "best_plantvillage_mixed_upsampled.pt"),
        "--data-dir", str(LOCAL_DATA_DIR),
        "--dataset", dataset,
        "--split", "test",
        "--results", str(RESULTS_DIR / "ablation_results.csv"),
        "--variant", "mixed_upsampled",
    ] + extra
    result = subprocess.run(cmd, cwd=str(REPO_DIR))
    assert result.returncode == 0, f"ml.evaluate failed for {dataset}"

## Step 6 - The verdict: did oversampling lift PlantDoc without losing the lab?

**Two gates** (must both pass):
1. **PlantDoc improved:** `mixed_upsampled` F1 on `plantdoc_test` > 0.1116 baseline (and ideally past
   `mixed`'s 0.4107, toward `both`'s 0.5578).
2. **No forgetting:** PlantVillage F1 >= 90% of the 0.9501 baseline (>= ~0.855).

The full comparison matters here: how `mixed_upsampled` sits relative to `mixed` (no oversampling)
and `both` (field-only, forgetting) on both datasets.


In [ ]:
import pandas as pd

df = pd.read_csv(str(RESULTS_DIR / "ablation_results.csv")).drop_duplicates()
cols = ["variant", "dataset", "accuracy", "precision", "recall", "f1"]
print(df[cols].to_string(index=False))

key = lambda v, d: df[(df["variant"] == v) & (df["dataset"] == d)]
base_pd = key("baseline_pv_only_no_aug", "plantdoc_test")
base_pv = key("baseline_pv_only_no_aug", "plantvillage_test")
ups_pd = key("mixed_upsampled", "plantdoc_test")
ups_pv = key("mixed_upsampled", "plantvillage_test")

print("\nThe single-model landscape (PlantDoc | PlantVillage):")
for label, pd_row, pv_row in [
    ("baseline       ", base_pd, base_pv),
    ("both (fine-tune)", key("both", "plantdoc_test"), key("both", "plantvillage_test")),
    ("mixed (5%)     ", key("mixed", "plantdoc_test"), key("mixed", "plantvillage_test")),
    ("mixed_upsampled", ups_pd, ups_pv),
]:
    pd_f1 = f"{pd_row.iloc[0]['f1']:.4f}" if not pd_row.empty else "  -  "
    pv_f1 = f"{pv_row.iloc[0]['f1']:.4f}" if not pv_row.empty else "  -  "
    print(f"  {label}  PlantDoc F1 {pd_f1} | PlantVillage F1 {pv_f1}")

if not ups_pd.empty and not ups_pv.empty and not base_pd.empty and not base_pv.empty:
    g1, b1 = base_pd.iloc[0]['f1'], base_pv.iloc[0]['f1']
    ok1 = ups_pd.iloc[0]['f1'] > g1
    ok2 = ups_pv.iloc[0]['f1'] >= 0.9 * b1
    print(f"\n  gate 1 PlantDoc F1 > {g1:.4f}:  {'PASS' if ok1 else 'FAIL'}")
    print(f"  gate 2 PlantVillage F1 >= {0.9 * b1:.4f}: {'PASS' if ok2 else 'FAIL'}")
    mixed_pd = key("mixed", "plantdoc_test")
    if not mixed_pd.empty:
        print(f"  vs mixed (no oversampling): PlantDoc F1 {ups_pd.iloc[0]['f1'] - mixed_pd.iloc[0]['f1']:+.4f}")
    if ok1 and ok2:
        print("\nSprint 6 DONE - oversampling kept both domains and pushed PlantDoc up.")
    else:
        print("\nSprint 6 NOT met yet - see which gate failed (try higher --plantdoc-repeat).")
else:
    print("\nMissing rows - make sure Sprints 4-5 ran and their CSV is in RESULTS_DIR.")

## Step 7 - Predict field + lab photos (done-when)

Classify a couple of field photos and lab photos with the best oversampled model - a correct PlantDoc
label means the bigger field share translated into real field predictions.


In [ ]:
for folder in (LOCAL_DATA_DIR / "plantdoc" / "test", LOCAL_DATA_DIR / "plantvillage" / "test"):
    images = sorted(folder.glob("*/*.jpg"))[:2]
    for image in images:
        cmd = [
            sys.executable, "-m", "ml.predict",
            "--checkpoint", str(CHECKPOINT_DIR / "best_plantvillage_mixed_upsampled.pt"),
            "--image", str(image),
            "--topk", "3",
        ]
        subprocess.run(cmd, cwd=str(REPO_DIR))
        print("  (true class folder:", image.parent.name, ")")

## Where things live

**On Google Drive (durable):**
```
folium/checkpoints/best_plantvillage_mixed_upsampled.pt   variant 7 (mixed_upsampled)
folium/results/ablation_results.csv                       all rows (the paper's source of truth)
folium/results/cm_mixed_upsampled.png                     PlantDoc + PlantVillage matrices
```
Every number in the CSV comes from an actual logged run - nothing fabricated.
